###DATA2-332
###STEPS
* Re-run the table `p-cdh`.derivative.dportfoliotoolsecuritized for dates > 2024-09-23 until today.

##

###DD-188
Create the following workflow in production env based on yaml :

resources:
  jobs:
    AML_Report:
      name: AML Report
      schedule:
        quartz_cron_expression: 0 0 19 ? * MON-FRI
        timezone_id: Europe/Athens
        pause_status: UNPAUSED
      tasks:
        - task_key: AML_Report
          notebook_task:
            notebook_path: AML Report/AML_Report
            source: GIT
      git_source:
        git_url: https://git-codecommit.eu-central-1.amazonaws.com/v1/repos/gr.cepal.dna.de.delivery
        git_provider: awsCodeCommit
        git_branch: master
      queue:
        enabled: true

###D360-1829

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.paymentschedule
ADD COLUMN ImplementationFlag boolean AFTER DownPaymentFlag;

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativearchive.dpaymentschedule
ADD COLUMN ImplementationFlag boolean AFTER DownPaymentFlag;

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativeactive.dpaymentschedule
ADD COLUMN ImplementationFlag boolean AFTER DownPaymentFlag;

###p-cadastral

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.realestatebuilding 
ADD COLUMN SourceTableName STRING AFTER RealEstateStatus;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `p-cadastral`.res;

In [0]:
%sql
CREATE TABLE `p-cadastral`.res.cadastre_persons
DEEP CLONE `qa-cadastral`.res.cadastre_persons;

In [0]:
%sql
TRUNCATE TABLE `p-cadastral`.res.cadastre_persons

In [0]:
%sql
CREATE TABLE `p-cadastral`.res.cadastre_properties
DEEP CLONE `qa-cadastral`.res.cadastre_properties;

In [0]:
%sql
TRUNCATE TABLE `p-cadastral`.res.cadastre_properties

In [0]:
%sql
CREATE OR REPLACE TABLE `p-cdh`.reference.lookup_export AS
SELECT * FROM `qa-cdh`.reference.lookup_export ;

In [0]:
%sql
UPDATE `p-cdh`.reference.lookup_export
SET Catalog = '`p-cdh-delivery`' ;

###Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



###Closed loans function update

In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `p-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0
AND B.JustificationCode != 43023

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')